# Week 1 — Day 2 Exercise Solution: Ollama-based Summarizer

Same project as Day 1 (webpage summarizer) but swapped over to a local open-source model running via Ollama instead of the OpenAI API.

**Why this matters:** the same technique works for every project later in the course if you'd rather not pay for API calls.

**Trade-offs:**
- ✅ No API charges — open-source
- ✅ Data never leaves the local machine
- ❌ Significantly less capable than frontier models (GPT-4o, Claude, Gemini)

**Setup:** install Ollama from https://ollama.com. Once installed, the server runs locally — visit http://localhost:11434/ to confirm it shows `Ollama is running`. If not, run `ollama serve` in a Terminal/Powershell.

In [ ]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [ ]:
MODEL = "llama3.2"

## Website scraper class

In [ ]:
class Website:
    """A utility class to represent a Website that we have scraped."""

    url: str
    title: str
    text: str

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
site = Website("https://anthropic.com")
print(site.title)
print(site.text[:500])

## Prompts

Same system + user prompt structure as Day 1. The Ollama API expects the same message format as OpenAI:

```python
[
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."}
]
```

In [ ]:
system_prompt = (
    "You are an assistant that analyzes the contents of a website "
    "and provides a short summary, ignoring text that might be navigation related. "
    "Respond in markdown."
)

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += (
        "\nThe contents of this website is as follows; "
        "please provide a short summary of this website in markdown. "
        "If it includes news or announcements, then summarize these too.\n\n"
    )
    user_prompt += website.text
    return user_prompt

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## Summarize — using the `ollama` Python package directly

In [ ]:
def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

def display_summary(url):
    display(Markdown(summarize(url)))

In [ ]:
display_summary("https://anthropic.com")

In [ ]:
display_summary("https://cnn.com")

## Notes

- Won't work on JavaScript-rendered sites (React apps etc.) — would need Selenium or Playwright.
- CloudFront-protected sites may return 403.